## This script gets the sequences of ref and alt variants from the design file

### Top results checking for TFs
- identify the sequences without adapters from fasta file (header.str.contains('id von hit'))
- split the found header by "_fwd" and get the first part
- replace ALT_ with REF_ and get the sequence as reference

In [6]:
import pandas as pd
import yaml 

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# config 
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [24]:
design_fa = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])
variant_table = pd.read_csv(config['files']['final_design']['variant_table'], sep="\t")

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


In [9]:
high_effect_variants = pd.read_csv('/home/kisa/coding/80K_MPRA/80K-Analysis/06_variant_analysis/notebooks/high_effect_variants_resequencing.tsv', sep="\t")

,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id,Variant_without_group,CHROM,POS,...,QUAL,FILTER,INFO,DNase_max,max_col,enformer_variant_info,enformer_class_list,gene_set_list,variant_type_list,abs_logFC
0,1.521560,0.992996,13.599182,1.398939e-35,2.566633e-31,68.719277,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,TRIO|ENSG00000038382.23|EH38E3627037|5-1425991...,chr5,14259916,...,1,PASS,AF=0.0168281;AC=2558,NaN,NaN,"[('neuro', 'rare', 'NA')]",['NA'],['neuro'],['rare'],1.521560
1,1.482729,1.893987,19.158362,2.947206e-66,1.081448e-61,138.643345,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,TRIO|ENSG00000038382.23|EH38E2358394|5-1440805...,chr5,14408059,...,1,PASS,AF=6.57117e-06;AC=1,496.1264,109_DNASE:fibroblast of,"[('neuro', 'singleton', 'enformer_high')]",['enformer_high'],['neuro'],['singleton'],1.482729
2,1.221552,0.705690,6.746849,9.428517e-11,4.739315e-08,13.690928,cardiac_neuro_cava_random:LZTR1|ENSG0000009994...,LZTR1|ENSG00000099949.22|EH38E3469088|22-21014...,chr22,21014949,...,1,PASS,AF=1.31479e-05;AC=2,608.5865,"177_DNASE:CD8-positive,","[('neuro', 'ultra-rare', 'enformer_high')]",['enformer_high'],['neuro'],['ultra-rare'],1.221552
3,1.213732,0.644855,8.105816,1.155920e-14,1.211866e-11,22.197136,cardiac_neuro_cava_random:MDH2|ENSG00000146701...,MDH2|ENSG00000146701.12|EH38E3781047|7-7610916...,chr7,76109162,...,1,PASS,AF=0.000854735;AC=130,513.9389,123_DNASE:K562,"[('neuro', 'ultra-rare', 'enformer_high')]",['enformer_high'],['neuro'],['ultra-rare'],1.213732
4,1.184141,0.947644,8.744674,1.074335e-16,1.516217e-13,26.801432,cardiac_neuro_cava_random:DALRD3|ENSG000001781...,DALRD3|ENSG00000178149.17|EH38E2200024|3-49025...,chr3,49025606,...,1,PASS,AF=6.57004e-06;AC=1,639.2484,493_DNASE:common myeloid,"[('neuro', 'singleton', 'enformer_high')]",['enformer_high'],['neuro'],['singleton'],1.184141


In [37]:
top_2 = high_effect_variants.head(2)
bottom_2 = high_effect_variants.tail(2)
interesting_variants = pd.concat([top_2, bottom_2])

In [31]:
def write_ref_alt_sequences(sequence_df, output_path, header_cols, sequence_cols):
    """
    Write the fasta file with the header and sequence of the alterantive and reference sequences
    """
    with open(output_path, 'w') as f:
        for index, row in sequence_df.iterrows():
            f.write('>' + row[header_cols[0]] + '\n' + row[sequence_cols[0]] + '\n')
            f.write('>' + row[header_cols[1]] + '\n' + row[sequence_cols[1]] + '\n')
    return True


In [42]:
ids = interesting_variants['Variant_without_group'].to_list()
print(ids)
# replace | with \| for using contains

# get row in design fasta
escaped_ids = []
for elem in ids:
    print(elem)
    escaped_ids.append(elem.replace('|', '\|'))
    
alternative_sequences = design_fa.loc[design_fa['header'].str.contains('|'.join(escaped_ids))]
alternative_sequences.columns = ['ALT_ID', 'alt_sequence']

# # find references with variant table
alt_and_ref_id = alternative_sequences.merge(variant_table, on='ALT_ID', how='left')

# # get sequences for references as well
alt_and_ref_sequences = alt_and_ref_id.merge(design_fa, left_on='REF_ID', right_on='header')
alt_and_ref_sequences.rename(columns={'sequence': 'ref_sequence'}, inplace=True)
alt_and_ref_sequences
# remove adapters of coloms containing sequence
alt_and_ref_sequences['ref_sequence'] = alt_and_ref_sequences['ref_sequence'].str[15:285]
alt_and_ref_sequences['alt_sequence'] = alt_and_ref_sequences['alt_sequence'].str[15:285]

output_file = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/IGVF_Y1_design/projects/80K_MPRA/variant_tf_search/top_bottom_2_variants.fa'
output_file = '/home/kisa/coding/80K_MPRA/80K-Analysis/06_variant_analysis/notebooks/test_alt_ref_output.fa'
write_ref_alt_sequences(alt_and_ref_sequences, output_path=output_file, header_cols=['ALT_ID', 'REF_ID'], sequence_cols=['alt_sequence', 'ref_sequence'])

['TRIO|ENSG00000038382.23|EH38E3627037|5-14259916-C-T', 'TRIO|ENSG00000038382.23|EH38E2358394|5-14408059-A-G', 'PNKP|ENSG00000039650.12|EH38E3313885|19-49907159-A-C', 'TLK2|ENSG00000146872.19|EH38E1876701|17-62454069-C-A']
TRIO|ENSG00000038382.23|EH38E3627037|5-14259916-C-T
TRIO|ENSG00000038382.23|EH38E2358394|5-14408059-A-G
PNKP|ENSG00000039650.12|EH38E3313885|19-49907159-A-C
TLK2|ENSG00000146872.19|EH38E1876701|17-62454069-C-A


True